[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-2-ml-dl-essentials/01-what-is-ml/code/ml_basics.ipynb)

# Class 2.1: What is machine learning?

The slides carry the ideas. Here you run the loop end to end on real datasets that ship with scikit-learn, so nothing downloads.

What we will cover:
- a regression task (predict a number) and a classification task (predict a category)
- the three-way split, and picking a model on validation before touching the test set
- honest metrics: accuracy, precision, recall, and the confusion matrix
- the transforms you apply to data, and the leakage trap they can cause
- why accuracy alone lies on imbalanced data, and overfitting seen live


## Setup

You need scikit-learn for this notebook (numpy and pandas are already installed from Module 1). To install it in your virtual environment, run:

```
pip install scikit-learn
```


## Task type 1: regression predicts a number

`load_diabetes` gives 10 features per patient; the label is a continuous disease-progression score. We fit a line, predict, and measure the average distance between prediction and truth (mean absolute error).

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

d = load_diabetes()

d.data  # X : features

array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
         0.01990749, -0.01764613],
       [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
        -0.06833155, -0.09220405],
       [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
         0.00286131, -0.02593034],
       ...,
       [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
        -0.04688253,  0.01549073],
       [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
         0.04452873, -0.02593034],
       [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
        -0.00422151,  0.00306441]], shape=(442, 10))

In [ ]:
d.target   # y : target variable

array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
        69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
        68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
        87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
       259.,  53., 190., 142.,  75., 142., 155., 225.,  59., 104., 182.,
       128.,  52.,  37., 170., 170.,  61., 144.,  52., 128.,  71., 163.,
       150.,  97., 160., 178.,  48., 270., 202., 111.,  85.,  42., 170.,
       200., 252., 113., 143.,  51.,  52., 210.,  65., 141.,  55., 134.,
        42., 111.,  98., 164.,  48.,  96.,  90., 162., 150., 279.,  92.,
        83., 128., 102., 302., 198.,  95.,  53., 134., 144., 232.,  81.,
       104.,  59., 246., 297., 258., 229., 275., 281., 179., 200., 200.,
       173., 180.,  84., 121., 161.,  99., 109., 115., 268., 274., 158.,
       107.,  83., 103., 272.,  85., 280., 336., 281., 118., 317., 235.,
        60., 174., 259., 178., 128.,  96., 126., 28

In [3]:
Xr, yr = d.data, d.target
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.2, random_state=0)

reg = LinearRegression().fit(Xr_tr, yr_tr)  # Fitting linear regression model on training data
pred = reg.predict(Xr_te)  # Predicting on test data

print("test MAE (avg distance off):", round(mean_absolute_error(yr_te, pred), 1))
print("one patient  ->  predicted:", round(float(pred[0]), 1),
      " true:", round(float(yr_te[0]), 1),
      " off by:", round(abs(float(pred[0]) - float(yr_te[0])), 1))

test MAE (avg distance off): 46.2
one patient  ->  predicted: 238.5  true: 321.0  off by: 82.5


## Task type 2: classification predicts a category

`load_breast_cancer` describes each tumor with 30 numeric features; the label is malignant (0) or benign (1). Two classes, so this is binary classification. `X` is what the model sees, `y` is the answer it must produce.

In [4]:
from sklearn.datasets import load_breast_cancer

b = load_breast_cancer()
Xb, yb = b.data, b.target

print("feature matrix X:", Xb.shape)
print("labels y:", yb.shape)
print("classes:", {str(name): int((yb == i).sum()) for i, name in enumerate(b.target_names)})
print("row 0 label:", int(yb[0]), "->", b.target_names[yb[0]])

feature matrix X: (569, 30)
labels y: (569,)
classes: {'malignant': 212, 'benign': 357}
row 0 label: 0 -> malignant


## Split three ways: train, validation, test

Split off the test set first, then split the rest into train and validation. The model learns on train, you compare choices on validation, and the test set stays sealed until the very end.

In [5]:
X_tv, X_te, y_tv, y_te = train_test_split(
    Xb, yb, test_size=0.2, random_state=0, stratify=yb)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=0, stratify=y_tv)   # 0.25 of 0.8 = 0.2

print("train:", X_tr.shape[0], " validation:", X_val.shape[0], " test:", X_te.shape[0])

train: 341  validation: 114  test: 114


## Scale on train only, then choose a model on validation

The scaler is a transform that learns the mean and spread. Fit it on the training rows alone, or test information leaks in. Then train two candidates and let validation pick the winner. This is the tune loop from the process diagram.

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

scaler = StandardScaler().fit(X_tr)                 # fit on TRAIN only
X_tr_s, X_val_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_val), scaler.transform(X_te)

candidates = {
    "logistic regression": LogisticRegression(max_iter=5000).fit(X_tr_s, y_tr),
    "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr_s, y_tr),
}
val_acc = {name: accuracy_score(y_val, m.predict(X_val_s)) for name, m in candidates.items()}
for name, a in val_acc.items():
    print(f"validation accuracy  {name:26s} {a:.3f}")

best_name = max(val_acc, key=val_acc.get)
best = candidates[best_name]
print("chosen:", best_name)

validation accuracy  logistic regression        0.991
validation accuracy  decision tree (depth 3)    0.930
chosen: logistic regression


## The honest number: score the test set once

In [7]:
y_hat = best.predict(X_te_s)
print("TEST accuracy (the number that counts):", round(accuracy_score(y_te, y_hat), 3))

TEST accuracy (the number that counts): 0.974


## Better metrics: precision, recall, confusion matrix

Accuracy hides which mistakes happen. Treat malignant (0) as the positive class we must not miss. Precision asks how many flagged malignant really were; recall asks how many of the true malignant cases we caught.

In [8]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

print("precision (malignant):", round(precision_score(y_te, y_hat, pos_label=0), 3))
print("recall    (malignant):", round(recall_score(y_te, y_hat, pos_label=0), 3))
print("confusion matrix (rows = true 0/1, cols = predicted 0/1):")
print(confusion_matrix(y_te, y_hat))

precision (malignant): 0.976
recall    (malignant): 0.952
confusion matrix (rows = true 0/1, cols = predicted 0/1):
[[40  2]
 [ 1 71]]


## Multiclass: one of several categories

`load_wine` has three classes, so the model outputs a score per class and picks the highest. The setup is identical; only the number of classes changed.

In [9]:
from sklearn.datasets import load_wine

w = load_wine()
Xw, yw = w.data, w.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=0, stratify=yw)
scw = StandardScaler().fit(Xw_tr)
mc = LogisticRegression(max_iter=5000).fit(scw.transform(Xw_tr), yw_tr)

print("classes:", list(map(str, w.target_names)))
print("test accuracy:", round(accuracy_score(yw_te, mc.predict(scw.transform(Xw_te))), 3))

classes: ['class_0', 'class_1', 'class_2']
test accuracy: 1.0


## The transforms you will apply, up close

Three transforms show up on almost every dataset. Each one learns something from the data it is fit on: the mean and spread, the category list, or the fill value.

In [10]:
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer

# scale: put a column on a common scale (mean 0, spread 1)
col = np.array([[10.], [20.], [30.]])
print("scaled:", StandardScaler().fit_transform(col).ravel().round(2))

# encode: turn text categories into 0/1 columns
colors = pd.DataFrame({"color": ["red", "green", "blue", "red"]})
print(pd.get_dummies(colors, columns=["color"]).astype(int).to_string(index=False))

# impute: fill a missing value with the column median
vals = np.array([[1.], [np.nan], [3.], [5.]])
print("imputed:", SimpleImputer(strategy="median").fit_transform(vals).ravel())

scaled: [-1.22  0.    1.22]
 color_blue  color_green  color_red
          0            0          1
          0            1          0
          1            0          0
          0            0          1
imputed: [1. 3. 3. 5.]


## Trap 1: data leakage

Fit a transform on the whole dataset before splitting and it has already seen the test rows. The learned means differ, which is the leak in numbers. Split first, fit on train only.

In [11]:
leaked = StandardScaler().fit(Xb)      # saw everything, including test rows
honest = StandardScaler().fit(X_tr)    # saw only training rows
print("leaked mean[0]:", round(float(leaked.mean_[0]), 3))
print("honest mean[0]:", round(float(honest.mean_[0]), 3))
print("they differ, so the leaked scaler encodes test information")

leaked mean[0]: 14.127
honest mean[0]: 14.103
they differ, so the leaked scaler encodes test information


## Trap 2: accuracy lies on imbalanced data

Make malignant rare, then a model that always guesses the majority scores high on accuracy and catches none of the rare cases. Recall exposes it.

In [13]:
from sklearn.dummy import DummyClassifier

mask = (yb == 1) | (np.arange(len(yb)) % 15 == 0)   # all benign + a few malignant
Xi, yi = Xb[mask], yb[mask]
print("class count:", {int(c): int((yi == c).sum()) for c in (0, 1)})

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
dummy = DummyClassifier(strategy="most_frequent").fit(Xi_tr, yi_tr)
p = dummy.predict(Xi_te)
print("accuracy         :", round(accuracy_score(yi_te, p), 3))
print("recall(malignant):", round(recall_score(yi_te, p, pos_label=0), 3), " it caught none")

class count: {0: 13, 1: 357}
accuracy         : 0.964
recall(malignant): 0.0  it caught none


## Overfitting, live

An unrestricted tree memorizes the training rows: perfect on train, worse on test. A shallow tree generalizes better. Watch the train-versus-test gap.

In [14]:
deep = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)             # no depth limit

shallow = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)

for name, m in [("deep (overfit)", deep), ("shallow", shallow)]:
    print(f"{name:16s} train {m.score(X_tr, y_tr):.3f}   test {m.score(X_te, y_te):.3f}")

deep (overfit)   train 1.000   test 0.921
shallow          train 0.962   test 0.930


## Your turn

**Micro-assignment.** Six problems on the ML setup; see `../micro-assignment/README.md`.

**Next, class 2.2 (How models learn):** loss, the gradient, and the downhill step that actually fits the line.